# Multiband parametric fit: RAINBOW model

[`RainbowFit`](../api/#light_curve.RainbowFit) fits a physical **blackbody + bolometric-light-curve** model to multi-band
photometric time series. It is designed for extragalactic transients such as supernovae,
kilonovae, or tidal disruption events, where the SED is well approximated by a Planck
function whose temperature and luminosity evolve with time.

The method is described in [Russeil et al. 2024](https://ui.adsabs.harvard.edu/abs/2024A%26A...683A.251R/abstract).

## Prerequisites

[`RainbowFit`](../api/#light_curve.RainbowFit) requires [`iminuit`](https://iminuit.readthedocs.io/) ≥ 2.21.
Install it together with all optional dependencies or separately.
We would also need `matplotlib` for plotting, but of course it is not needed for the fitting itself.

In [ ]:
# %pip install "light-curve[full]" matplotlib
# OR
# %pip install light-curve iminuit matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from light_curve import RainbowFit

## Generating synthetic supernova data

We simulate a Bazin-shaped bolometric light curve with a sigmoid temperature evolution,
observed in four LSST-like bands (*g r i z*). Observations are flux densities — not
magnitudes — because `RainbowFit` works in linear flux space.

The sigmoid temperature is parametrized by the **peak temperature** `T` (the early, hot
plateau) and a **ratio** `T_ratio` = $T_\text{min}/T_\text{max}$ (the late, cool floor as a
fraction of the peak). Here the source cools from 15 000 K to 5 000 K, i.e. `T` = 15 000 K
and `T_ratio` = 1/3.

### Physical parameters

| Parameter | Symbol | Value |
|-----------|--------|-------|
| Peak time | $t_0$ | 60 000 (MJD) |
| Amplitude | $A$ | 1.0 |
| Rise time | $t_\text{rise}$ | 5 d |
| Fall time | $t_\text{fall}$ | 30 d |
| Peak temperature | $T$ | 15 000 K |
| Temperature ratio | $T_\text{ratio} = T_\text{min}/T_\text{max}$ | 1/3 (i.e. $T_\text{min}$ = 5 000 K) |
| Colour timescale | $t_\text{color}$ | 10 d |

In [ ]:
rng = np.random.default_rng(0)

# Effective wavelengths in Angstrom for griz
band_wave_aa = {"g": 4770.0, "r": 6231.0, "i": 7625.0, "z": 9134.0}

# True parameters
reference_time = 60_000.0
amplitude      = 1.0
rise_time      = 5.0
fall_time      = 30.0
Tmax           = 15_000.0  # peak (early, hot) temperature -> fit parameter `T`
Tmin           = 5_000.0   # late (cool) floor; enters the fit as the ratio T_ratio = Tmin / Tmax
t_color        = 10.0

# Per-band baseline flux (constant offset)
baselines = {b: 0.3 * amplitude + rng.exponential(scale=0.3 * amplitude) for b in band_wave_aa}

# Build the feature object first so we can use .model() to generate data
feature = RainbowFit.from_angstrom(
    band_wave_aa,
    with_baseline=True,
    bolometric="bazin",
    temperature="sigmoid",
)

# True parameter vector, in feature.names order:
# common + bolometric + temperature + baselines + chi2_placeholder
true_params = [
    reference_time, amplitude, rise_time, fall_time,  # Bazin bolometric
    Tmax, Tmin / Tmax, t_color,                        # sigmoid temperature: T (peak), T_ratio, t_color
    *baselines.values(),                               # per-band baselines
    1.0,                                               # reduced chi^2 placeholder
]

# Random observation times covering ±3 rise/fall windows
n_obs = 1000
t    = np.sort(rng.uniform(reference_time - 3 * rise_time, reference_time + 3 * fall_time, n_obs))
band = rng.choice(list(band_wave_aa), size=n_obs)

# True model flux
flux = feature.model(t, band, *true_params)

# Poisson-like noise: S/N=10 at minimum flux
flux_err = np.sqrt(flux * np.min(flux)) / 10.0
flux = flux + rng.normal(0.0, flux_err)

print(f"Generated {n_obs} observations in bands {list(band_wave_aa)}")

## Fitting the light curve

Call `feature(t, flux, sigma=flux_err, band=band)`. The result is a NumPy array of
fit parameters followed by the **reduced χ²** of the fit.

In [ ]:
result = feature(t, flux, sigma=flux_err, band=band)

# Parameter names are stored in feature.names
print("Parameter names:", feature.names)
print()
for name, val, true in zip(feature.names, result, true_params):
    print(f"  {name:25s}  fitted={val:10.3f}  true={true:10.3f}")

## Visualising the fit

Plot the data and the best-fit model curve for each band.

In [ ]:
colors = {"g": "#2ca02c", "r": "#d62728", "i": "#9467bd", "z": "#8c564b"}

fig, ax = plt.subplots(figsize=(9, 4))

t_grid = np.linspace(t.min(), t.max(), 400)

for b in band_wave_aa:
    mask = band == b
    ax.errorbar(
        t[mask], flux[mask], yerr=flux_err[mask],
        fmt=".", color=colors[b], alpha=0.4, label=f"{b} data",
    )
    band_grid = np.full(len(t_grid), b)
    ax.plot(t_grid, feature.model(t_grid, band_grid, *result), color=colors[b], lw=2)

ax.set_xlabel("Time (MJD)")
ax.set_ylabel("Flux density")
ax.set_title("RainbowFit — Bazin bolometric, sigmoid temperature")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

## Parameter uncertainties

`fit_and_get_errors` returns a `(params, errors)` tuple. The errors are estimated by
`iminuit` from the Hessian of the chi-squared surface.

In [ ]:
params, errors = feature.fit_and_get_errors(t, flux, sigma=flux_err, band=band)

print(f"{'Parameter':25s}  {'Value':>10s}  {'Error':>10s}  {'True':>10s}")
print("-" * 62)
for name, val, err, true in zip(feature.names, params, errors, true_params):
    print(f"{name:25s}  {val:10.3f}  {err:10.3f}  {true:10.3f}")

## Peak time

The bolometric peak time can differ from `reference_time` for asymmetric light curves.
`feature.peak_time(params)` computes it analytically from the bolometric term.

In [ ]:
t_peak = feature.peak_time(params)
print(f"Fitted peak time : {t_peak:.2f} MJD")
print(f"True   peak time : {feature.peak_time(np.array(true_params)):.2f} MJD")

## Choosing different models

`RainbowFit` accepts pluggable **bolometric**, **temperature**, and **spectral** terms.

### Bolometric terms

| Name | Parameters | Description |
|------|-----------|-------------|
| `'bazin'` (default) | `reference_time, amplitude, rise_time, fall_time` | Bazin 2011 rise-and-fall |
| `'sigmoid'` | `reference_time, amplitude, rise_time` | Logistic rise only |
| `'linexp'` | `reference_time, amplitude, rise_time` | Linear rise, exponential decay |
| `'doublexp'` | `reference_time, amplitude, time1, time2, p` | Exponential rise, decay with two exponentials |

### Temperature terms

The sigmoid terms use a peak temperature `T` and a ratio `T_ratio` = $T_\text{min}/T_\text{max}$
(`T_ratio` = 1 is a constant temperature; a weak prior anchors it there unless the data demand cooling).

| Name | Parameters | Description |
|------|-----------|-------------|
| `'sigmoid'` (default) | `T, T_ratio, t_color` | Temperature drops sigmoidally from the peak `T` to `T·T_ratio` |
| `'constant'` | `T` | Fixed temperature |
| `'delayed_sigmoid'` | `T, T_ratio, t_color, t_delay` | Sigmoid delayed relative to the bolometric peak |

### Spectral terms

`'planck'` is a pure blackbody. The other terms describe SEDs that *deviate* from a blackbody
(UV blanketing or power-law spectra); each reduces to Planck at a **null** parameter value and
carries a weak Gaussian prior anchoring it there, so a true blackbody is recovered unbiased.

| Name | Parameters | Description |
|------|-----------|-------------|
| `'planck'` (default) | *(none)* | Standard blackbody $B_\nu(T)$ — no extra parameters |
| `'blanketed'` | `lambda_scale` | UV-extincted blackbody $B_\nu(T)\,e^{-\tau}$, $\tau = I\,e^{-\lambda/\lambda_s}$; the extinction reach $\lambda_s$ is anchored to the characteristic temperature `T` (shared with the temperature term) so the blanketing depth stays fixed as the source cools |
| `'modified_bb'` | `beta` | Planck tilted by a power law, $B_\nu(T)\,(\lambda/\lambda_\text{ref})^{\beta}$; $\beta = 0 \Rightarrow$ Planck |
| `'logparabola'` | `sp_a, sp_b` | Planck × $e^{a L + b L^2}$, $L = \ln(\lambda/\lambda_\text{ref})$; $a = b = 0 \Rightarrow$ Planck |
| `'genwien'` | `spec_k` | Generalized Wien $B_\nu \propto \nu^3 e^{-x^{\,\texttt{spec\_k}}}$, $x = h\nu/k_B T$; $\texttt{spec\_k} = 1 \Rightarrow$ Wien tail |

Both backends accept an `optimizer` argument: `"iminuit"` (default, robust Migrad) or
`"least_squares"` (scipy Trust Region Reflective, sharing the same analytic Jacobian — usually
faster, with an automatic fall-back to iminuit when needed).

In [ ]:
# Constant temperature — fewer parameters, useful for bluer transients
feature_const_T = RainbowFit.from_angstrom(
    band_wave_aa,
    with_baseline=False,
    bolometric="bazin",
    temperature="constant",
)
result_const = feature_const_T(t, flux, sigma=flux_err, band=band)
print("Parameters (constant T model):")
for name, val in zip(feature_const_T.names, result_const):
    print(f"  {name:25s}  {val:.3f}")

In [ ]:
# Blanketed blackbody — adds UV extinction parameters: lambda_scale
feature_blanketed = RainbowFit.from_angstrom(
    band_wave_aa,
    with_baseline=False,
    bolometric="bazin",
    temperature="sigmoid",
    spectral="blanketed",
)
result_blanketed = feature_blanketed(t, flux, sigma=flux_err, band=band)
print("Parameters (blanketed spectral model):")
for name, val in zip(feature_blanketed.names, result_blanketed):
    print(f"  {name:25s}  {val:.3f}")

### Non-blackbody SED terms and the fast optimizer

Beyond `'blanketed'`, the spectral terms `'modified_bb'`, `'logparabola'`, and `'genwien'`
describe SEDs that depart from a pure blackbody (a power-law tilt, a log-parabola, or a
generalized-Wien blue cutoff). Each reduces to Planck at a *null* parameter value
(`beta = 0`, `sp_a = sp_b = 0`, `spec_k = 1`), and a weak prior anchors it there — so fitting
them to genuinely blackbody data just recovers that null rather than inventing a deviation.

The fit itself can run on scipy's `least_squares` backend via `optimizer="least_squares"`,
which shares the analytic Jacobian of the default `iminuit` path and is usually faster.

In [ ]:
# Each deviation term reduces to a pure blackbody at a "null" parameter value, with a weak
# prior anchoring it there. Fitting them to the (blackbody) data above recovers that null —
# i.e. no spurious deviation is introduced.
shape_params = {"modified_bb": ["beta"], "logparabola": ["sp_a", "sp_b"], "genwien": ["spec_k"]}
for spectral, shape in shape_params.items():
    feat = RainbowFit.from_angstrom(
        band_wave_aa, with_baseline=True,
        bolometric="bazin", temperature="sigmoid", spectral=spectral,
    )
    res = feat(t, flux, sigma=flux_err, band=band)
    shape_vals = ", ".join(f"{p}={res[feat.names.index(p)]:+.3f}" for p in shape)
    print(f"{spectral:12s}  reduced chi2 = {res[-1]:.2f}   {shape_vals}")

# The same model fitted with the scipy least_squares backend (shares the analytic Jacobian).
feature_ls = RainbowFit.from_angstrom(
    band_wave_aa, with_baseline=True,
    bolometric="bazin", temperature="sigmoid", optimizer="least_squares",
)
result_ls = feature_ls(t, flux, sigma=flux_err, band=band)
print(f"\n{'least_squares':12s}  reduced chi2 = {result_ls[-1]:.2f}  (matches the default iminuit fit)")

## See also

- [RainbowFit API reference](../api/#light_curve.RainbowFit)
- [Feature table](../../) — all extractors
- [Parametric fits tutorial](../../fitting/) — single-band transient fits